In [35]:
#!python -m pip install --upgrade pip
#%pip install pandas matplotlib seaborn scikit-learn openpyxl tensorflow xgboost aif360
#%pip install "aif360[Reductions, inFairness]"

In [36]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pprint import pprint
from collections import Counter
from scipy.stats import chi2_contingency, fisher_exact

from fairlearn.metrics import MetricFrame
from fairlearn.metrics import demographic_parity_ratio, equalized_odds_ratio 
from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference, selection_rate, false_positive_rate, false_negative_rate, count
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

random_seed = 15

In [37]:
PATH = 'C:/Users/andre/Desktop/ProjectWork_AEQUITAS_AKKODIS/'

with open(PATH + 'data/predictions.json', 'r') as f:
    data = json.load(f)
predictions_df = pd.DataFrame(data['predictions'])
y_test = pd.Series(data['reference'])
s_test = pd.Series(data['sensitive'])
sensitive_features = data['sensitive_name']

s_test_dict = {feature: [row[i] for row in s_test] for i, feature in enumerate(sensitive_features)}
pprint(s_test_dict)

predictions_df.head(50)

{'Italian Residence': [1,
                       1,
                       1,
                       1,
                       1,
                       1,
                       1,
                       1,
                       1,
                       1,
                       1,
                       1,
                       1,
                       1,
                       1,
                       1,
                       1,
                       1,
                       1,
                       1,
                       1,
                       1,
                       1,
                       1,
                       1,
                       1,
                       1,
                       1,
                       1,
                       1,
                       1,
                       1,
                       1,
                       1,
                       0,
                       1,
                       1,
                       1,
            

,Logistic Regression_preprocessed_cr,Linear Regression_preprocessed_cr,Decision Tree_preprocessed_cr,Naive Bayes_preprocessed_cr,XGBoost_preprocessed_cr,KNN_preprocessed_cr,Neural Network_preprocessed_cr,Linear Regression_inprocessed_gfc,Logistic Regression_postprocessed_to,Linear Regression_postprocessed_to,Decision Tree_postprocessed_to,Naive Bayes_postprocessed_to,XGBoost_postprocessed_to,KNN_postprocessed_to,Neural Network_postprocessed_to
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
7,1,1,1,1,1,1,1,0,1,0,1,1,0,0,0
8,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
9,1,1,1,1,1,0,1,1,1,1,1,1,1,1,1


## Fairness Metrics

In [38]:
metrics = []
for name in predictions_df.columns:
    y_pred = predictions_df[name]
    accuracy = round(accuracy_score(y_test, y_pred), 3)
    precision = round(precision_score(y_test, y_pred), 3)
    recall = round(recall_score(y_test, y_pred), 3)
    f1 = round(f1_score(y_test, y_pred), 3)
    roc_auc = round(roc_auc_score(y_test, y_pred), 3)

    metrics.append({
        'Model': name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-score': f1,
        'ROC AUC': roc_auc
    })
metrics = pd.DataFrame(metrics)
print(metrics)

                                   Model  Accuracy  Precision  Recall  \
0    Logistic Regression_preprocessed_cr     0.983      0.984   0.968   
1      Linear Regression_preprocessed_cr     1.000      1.000   1.000   
2          Decision Tree_preprocessed_cr     1.000      1.000   1.000   
3            Naive Bayes_preprocessed_cr     0.973      0.940   0.989   
4                XGBoost_preprocessed_cr     1.000      1.000   1.000   
5                    KNN_preprocessed_cr     0.704      0.658   0.405   
6         Neural Network_preprocessed_cr     0.936      1.000   0.826   
7      Linear Regression_inprocessed_gfc     0.838      0.991   0.563   
8   Logistic Regression_postprocessed_to     0.971      0.983   0.937   
9     Linear Regression_postprocessed_to     0.988      0.989   0.979   
10        Decision Tree_postprocessed_to     0.996      1.000   0.989   
11          Naive Bayes_postprocessed_to     0.990      1.000   0.974   
12              XGBoost_postprocessed_to     0.986 

In [39]:
def compute_fairness_metrics(y_true, y_pred, s_test, sensitive_features, label=None):
    sf = pd.DataFrame(s_test.tolist(), columns=sensitive_features)
    mf = MetricFrame(
        metrics={
            'selection_rate': selection_rate,
            'fpr': false_positive_rate,
            'fnr': false_negative_rate,
            'count': count
        },
        y_true=y_true,
        y_pred=y_pred,
        sensitive_features=sf
    )

    dp_diff = demographic_parity_difference(y_true, y_pred, sensitive_features=sf)
    eo_diff = equalized_odds_difference(y_true, y_pred, sensitive_features=sf)

    dp = demographic_parity_ratio(y_true, y_pred, sensitive_features=sf)
    eo = equalized_odds_ratio(y_true, y_pred, sensitive_features=sf)

    if label:
        print(f"=== {label} ===")

    print("By group:")
    print(mf.by_group)
    print()
    print("Overall (selection_rate, fpr, fnr, count):")
    print(mf.overall)
    print()
    print(f"Demographic parity difference: {dp_diff:.4f}")
    print(f"Equalized odds difference:     {eo_diff:.4f}\n")
    print()
    print(f"Demographic parity ratio: {dp:.4f}")
    print(f"Equalized odds ratio:     {eo:.4f}\n")

    return mf

for name in predictions_df.columns:
    compute_fairness_metrics(y_test, predictions_df[name], s_test, sensitive_features, label=name)

=== Logistic Regression_preprocessed_cr ===
By group:
                       selection_rate       fpr       fnr  count
Sex Italian Residence                                           
0   0                        0.250000  0.000000  0.000000    4.0
    1                        0.440367  0.000000  0.020408  109.0
1   0                        0.214286  0.000000  0.400000   14.0
    1                        0.346154  0.011765  0.022222  390.0

Overall (selection_rate, fpr, fnr, count):
selection_rate      0.361702
fpr                 0.009174
fnr                 0.031579
count             517.000000
dtype: float64

Demographic parity difference: 0.2261
Equalized odds difference:     0.4000


Demographic parity ratio: 0.4866
Equalized odds ratio:     0.0000

=== Linear Regression_preprocessed_cr ===
By group:
                       selection_rate  fpr  fnr  count
Sex Italian Residence                                 
0   0                        0.250000  0.0  0.0    4.0
    1             

#### **3.1 Demographic Parity**

In [40]:
tolerance = 0.15
significance_level = 0.1

In [43]:
def calculate_demographic_parity(predictions, sensitive_attribute, name, significance_level, tolerance, activate_check=False):
    df = pd.DataFrame({
        'predictions': predictions,
        'sensitive_attribute': sensitive_attribute
    })
    prop = df.groupby('sensitive_attribute')['predictions'].mean()
    
    if activate_check:
        print(f"=== {name} ===")
        print(f"{prop}")

    if prop.shape[0] == 2:
        diff = prop.max() - prop.min()
        if activate_check:
            print(f"Two groups: |Δ| = {diff:.4f}, tol = {tolerance}")
        return 'T' if diff <= tolerance else False
    
    contingency_table = pd.crosstab(df['predictions'], df['sensitive_attribute'])
    chi2, p, dof, expected = chi2_contingency(contingency_table, correction=False)
    
    if contingency_table.shape == (2, 2) and (expected < 5).any():
        _, p = fisher_exact(contingency_table)
        if activate_check:
            print(f"Fisher’s exact test fallback for {name}")
    elif contingency_table.shape != (2, 2) and (expected < 5).any():
        if activate_check:
            print(f"Sparse contingency for {name}")
        
    return 'T' if p > significance_level else False    

results = {}
for sensitive_feature in sensitive_features:
    results[sensitive_feature] = []
    for name in predictions_df.columns:
        result = calculate_demographic_parity(predictions_df[name], s_test_dict[sensitive_feature], f"{name} ({sensitive_feature})", 
                                              significance_level, tolerance, activate_check=True)
        results[sensitive_feature].append(result)

sf_df = pd.DataFrame(results, index=predictions_df.columns)

=== Logistic Regression_preprocessed_cr (Sex) ===
sensitive_attribute
0    0.433628
1    0.341584
Name: predictions, dtype: float64
Two groups: |Δ| = 0.0920, tol = 0.15
=== Linear Regression_preprocessed_cr (Sex) ===
sensitive_attribute
0    0.442478
1    0.346535
Name: predictions, dtype: float64
Two groups: |Δ| = 0.0959, tol = 0.15
=== Decision Tree_preprocessed_cr (Sex) ===
sensitive_attribute
0    0.442478
1    0.346535
Name: predictions, dtype: float64
Two groups: |Δ| = 0.0959, tol = 0.15
=== Naive Bayes_preprocessed_cr (Sex) ===
sensitive_attribute
0    0.451327
1    0.368812
Name: predictions, dtype: float64
Two groups: |Δ| = 0.0825, tol = 0.15
=== XGBoost_preprocessed_cr (Sex) ===
sensitive_attribute
0    0.442478
1    0.346535
Name: predictions, dtype: float64
Two groups: |Δ| = 0.0959, tol = 0.15
=== KNN_preprocessed_cr (Sex) ===
sensitive_attribute
0    0.318584
1    0.200495
Name: predictions, dtype: float64
Two groups: |Δ| = 0.1181, tol = 0.15
=== Neural Network_preprocesse

#### **3.2 Equalized odds**

In [44]:
tolerance = 0.15
significance_level = 0.1

In [46]:
def calculate_equalized_odds(predictions, true_labels, sensitive_attribute, name, tolerance, activate_check=False):
    df = pd.DataFrame({
        'predictions': predictions,
        'true_labels': true_labels,
        'sensitive_attribute': sensitive_attribute
    })
    tprs, fprs = [], []
    for _, group_df in df.groupby('sensitive_attribute'):
        tn, fp, fn, tp = confusion_matrix(group_df['true_labels'], group_df['predictions'], labels=[0, 1]).ravel()
        tprs.append(tp / (tp + fn) if tp + fn != 0 else 0)
        fprs.append(fp / (fp + tn) if fp + tn != 0 else 0)

    max_tpr_diff = max(tprs) - min(tprs)
    max_fpr_diff = max(fprs) - min(fprs)

    if activate_check:
            print(f"=== {name} ===")
            print(f"Max FPR difference: {max_fpr_diff}")
            print(f"Max TPR difference: {max_tpr_diff}")

    return 'T' if (max_tpr_diff <= 2 * tolerance and max_fpr_diff <= 2 * tolerance) else False

results = {}
for sensitive_feature in sensitive_features:
    results[sensitive_feature] = []
    for name in predictions_df.columns:
        result = calculate_equalized_odds(predictions_df[name], y_test, s_test_dict[sensitive_feature], f"{name} ({sensitive_feature})", 
                                              tolerance, activate_check=True)
        results[sensitive_feature].append(result)

sf_df = pd.DataFrame(results, index=predictions_df.columns)

=== Logistic Regression_preprocessed_cr (Sex) ===
Max FPR difference: 0.011363636363636364
Max TPR difference: 0.01571428571428568
=== Linear Regression_preprocessed_cr (Sex) ===
Max FPR difference: 0.0
Max TPR difference: 0.0
=== Decision Tree_preprocessed_cr (Sex) ===
Max FPR difference: 0.0
Max TPR difference: 0.0
=== Naive Bayes_preprocessed_cr (Sex) ===
Max FPR difference: 0.013528138528138528
Max TPR difference: 0.040000000000000036
=== XGBoost_preprocessed_cr (Sex) ===
Max FPR difference: 0.0
Max TPR difference: 0.0
=== KNN_preprocessed_cr (Sex) ===
Max FPR difference: 0.0844155844155844
Max TPR difference: 0.10142857142857142
=== Neural Network_preprocessed_cr (Sex) ===
Max FPR difference: 0.0
Max TPR difference: 0.09999999999999998
=== Linear Regression_inprocessed_gfc (Sex) ===
Max FPR difference: 0.003787878787878788
Max TPR difference: 0.07714285714285718
=== Logistic Regression_postprocessed_to (Sex) ===
Max FPR difference: 0.027958152958152956
Max TPR difference: 0.104285